In [14]:
import os
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"

In [55]:
import os
import warnings
import numpy as np
import torch
import tensorflow as tf
import keras_hub
import librosa
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Alignment

tf.config.optimizer.set_jit(False) 
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
tf.config.set_visible_devices([], 'GPU')

In [ ]:
PRESET       = "whisper_medium_multi"
SAMPLE_RATE  = 16000
AUDIO_FOLDER = "./audio_files/"
INPUT_EXCEL  = "./AutoEIT Sample Audio for Transcribing.xlsx"
OUTPUT_EXCEL = "./AutoEIT Transcriptions.xlsx"

SKIP = {
    "038010_EIT-2A.mp3": 150,
    "038011_EIT-1A.mp3": 150,
    "038012_EIT-2A.mp3": 720,
    "038015_EIT-1A.mp3": 135,
}

SHEET = {
    "038010_EIT-2A.mp3": "38010-2A",
    "038011_EIT-1A.mp3": "38011-1A",
    "038012_EIT-2A.mp3": "38012-2A",
    "038015_EIT-1A.mp3": "38015-1A",
}
 
VAD_THRESHOLD = 0.6

EOT   = 50256
SOT   = 50258
ES    = 50262
TRANS = 50359
NOTSP = 50363
MAX_NEW_TOKENS = 100

In [82]:
from silero_vad import load_silero_vad
vad_model = load_silero_vad()

In [5]:
converter = keras_hub.layers.WhisperAudioConverter.from_preset(PRESET)
backbone  = keras_hub.models.WhisperBackbone.from_preset(PRESET)
tokenizer = keras_hub.tokenizers.WhisperTokenizer.from_preset(PRESET)

In [83]:
def load_audio(path, skip_sec):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    
    return audio[skip_sec * SAMPLE_RATE:]

In [84]:
def get_segments(audio, vad):
    from silero_vad import get_speech_timestamps

    tensor = torch.from_numpy(audio).float()
    timestamps = get_speech_timestamps(
        tensor, vad,
        threshold               = VAD_THRESHOLD,
        sampling_rate           = SAMPLE_RATE,
        min_silence_duration_ms = 3500,
        speech_pad_ms           = 300,
        return_seconds          = False,
    )

    segments = [
        (s["start"], s["end"])
        for s in timestamps
        if 0.65 <= (s["end"] - s["start"]) / SAMPLE_RATE <= 12.0
    ]

    return segments

In [85]:
import re
def clean_transcript(text):
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'\s+',     ' ', text)
    return text.strip()

In [86]:
def transcribe(audio_chunk, converter, backbone, tokenizer):

    audio_tf    = tf.constant(audio_chunk, dtype=tf.float32)
    spectrogram = converter(audio_tf)
    spectrogram = tf.expand_dims(spectrogram, 0)

    tokens     = [SOT, ES, TRANS, NOTSP]
    embeddings = backbone.token_embedding.embeddings
 
    for _ in range(MAX_NEW_TOKENS):
        decoder_input = tokens[-448:]
        decoder_ids   = tf.constant([decoder_input], dtype=tf.int32)
        padding_mask  = tf.ones_like(decoder_ids)

        outputs = backbone({
            "encoder_features":     spectrogram,
            "decoder_token_ids":    decoder_ids,
            "decoder_padding_mask": padding_mask,
        })

        hidden = outputs["decoder_sequence_output"] \
                 if isinstance(outputs, dict) else outputs

        logits     = tf.matmul(hidden[:, -1, :], embeddings, transpose_b=True)
        next_token = int(tf.argmax(logits, axis=-1).numpy()[0])
 
        if next_token >= EOT:
            break
        tokens.append(next_token)

    generated = [t for t in tokens[4:] if t < EOT]
    if not generated:
        return ""
 
    text = tokenizer.detokenize([generated])

    if hasattr(text, "numpy"):
        text = text.numpy()
    if isinstance(text, (list, np.ndarray)):
        text = text[0]
    if isinstance(text, bytes):
        text = text.decode("utf-8")
 
    return clean_transcript(str(text))

In [87]:
def process_file(path, skip_sec, vad, converter, backbone, tokenizer):
    print(f"\nFile: {os.path.basename(path)}")
    audio    = load_audio(path, skip_sec)
    segments = get_segments(audio, vad)
 
    results = []
    for i, (start, end) in enumerate(segments[:30]):
        MAX_SEC = 20
        chunk = audio[start : min(end, start + MAX_SEC * SAMPLE_RATE)]
        text  = transcribe(chunk, converter, backbone, tokenizer)
        results.append(text)
        print(f"  [{i+1:02d}] {text}")

    while len(results) < 30:
        results.append("")
    return results

In [88]:
def save_excel(all_results):
    wb    = load_workbook(INPUT_EXCEL)
    green = PatternFill(start_color="E2EFDA", end_color="E2EFDA", fill_type="solid")
    wrap  = Alignment(wrap_text=True, vertical="top")
 
    for filename, texts in all_results.items():
        ws = wb[SHEET[filename]]
        for row, text in enumerate(texts[:30], start=2):
            cell           = ws.cell(row=row, column=3, value=text)
            cell.fill      = green
            cell.alignment = wrap
 
    wb.save(OUTPUT_EXCEL)

In [ ]:
all_results = {}
 
for filename, skip_sec in SKIP.items():
    path = os.path.join(AUDIO_FOLDER, filename)
    if not os.path.exists(path):
        print(f"Not found: {path}")
        continue
    all_results[filename] = process_file(
        path, skip_sec, vad_model, converter, backbone, tokenizer
    )


File: 038010_EIT-2A.mp3
  [01] Quiero cortarme el pelo.
  [02] El libro está en la mesa.
  [03] El caro lo tiene pelo.
  [04] El se duche cada mañana.
  [05] Que dices ustedes que van a ser hoy?
  [06] Duro que sepa manejar muy bien.
  [07] Las calles de esta ciudad son muy anchas.
  [08] Puede que lleve mañana toda el día.
  [09] Las casas son muy bonitas pero caras.
  [10] Me gustan las películas que acaban bien.
  [11] El chico que hago es español.
  [12] Después de cenar, me fui a dormir tranquilo.
  [13] Quiero una casa en la que viven animales.
  [14] a nosotros
  [15] Ella solo bebe cerveza y no come nada.
  [16] Me gustaría...
  [17] Curso a la derecha y después siga recto.
  [18] Ella terminar pinta su apartamento
  [19] El niño a que se murió el gato. Triste.
  [20] una amiga mía cuidado los niños
  [21] El gato que era negro.
  [22] Antes de poder salir tiene que limpiar su cuarto.
  [23] la cantidad de personas que fuman
  [24] Después de llegar a casa
  [25] El ladrón pol

In [80]:
save_excel(all_results)